In [ ]:
import os, math, time, itertools, platform, torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, BertForMaskedLM,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments
)


# 1. Dataset (WT103, 5M tokens)

def build_wt103_subset(tokenizer, target_tokens=5_000_000, max_len=512, seed=1337):
    raw = load_dataset("wikitext", "wikitext-103-raw-v1")
    tok = tokenizer
    tok.model_max_length = 10_000_000

    taken_idx, seen = [], 0
    for i, line in enumerate(raw["train"]["text"]):
        if not line or not line.strip():
            continue
        seen += len(tok.tokenize(line))
        taken_idx.append(i)
        if seen >= target_tokens:
            break

    data = {
        "train": raw["train"].select(taken_idx),
        "validation": raw["validation"],
    }

    def tok_fn(batch):
        return tok(batch["text"], add_special_tokens=False,
                   truncation=False, padding=False,
                   return_attention_mask=False, return_token_type_ids=False)

    tokenized = {k: v.map(tok_fn, batched=True, remove_columns=["text"]) for k, v in data.items()}

    def group_and_add_specials(examples):
        all_ids = list(itertools.chain.from_iterable(examples["input_ids"]))
        eff_len = max_len - tok.num_special_tokens_to_add(pair=False)  
        total_len = (len(all_ids) // eff_len) * eff_len
        if total_len == 0:
            return {"input_ids": [], "attention_mask": [], "token_type_ids": []}
        chunks = [all_ids[i:i+eff_len] for i in range(0, total_len, eff_len)]
        input_ids = [tok.build_inputs_with_special_tokens(c) for c in chunks]
        attention_mask = [[1]*len(x) for x in input_ids]
        token_type_ids = [[0]*len(x) for x in input_ids]
        return {"input_ids": input_ids, "attention_mask": attention_mask, "token_type_ids": token_type_ids}

    for split in data.keys():
        tokenized[split] = tokenized[split].map(group_and_add_specials, batched=True,
                                                remove_columns=tokenized[split].column_names)

    tokenized["train"] = tokenized["train"].shuffle(seed=seed)
    return tokenized


# 2. Train baseline

def main():
    model_name = "bert-base-uncased"
    max_len = 512
    target_tokens = 5_000_000
    seed = 1337
    bs = 16
    max_steps = 20_000

    tok = AutoTokenizer.from_pretrained(model_name)
    tokenized = build_wt103_subset(tok, target_tokens, max_len, seed)

    collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm_probability=0.15)
    model = BertForMaskedLM.from_pretrained(model_name)

    torch.backends.cuda.matmul.allow_tf32 = True
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    args = TrainingArguments(
        output_dir="./results/baseline_wt103_5M",
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs,
        max_steps=max_steps,
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        eval_strategy="steps",
        eval_steps=1000,
        logging_strategy="steps",
        logging_steps=100,
        save_strategy="steps",
        save_steps=5000,
        seed=seed,
        remove_unused_columns=False,
        fp16=torch.cuda.is_available(),
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=args,
        data_collator=collator,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
    )

    start = time.time()
    trainer.train()
    wall = time.time() - start

    eval_res = trainer.evaluate()
    loss = float(eval_res["eval_loss"])
    ppl = math.exp(loss) if loss < 20 else float("inf")

    effective_tokens = max_steps * bs * max_len
    toks_per_sec = effective_tokens / max(wall, 1e-6)

    print(f"Baseline: Eval loss: {loss:.4f} | Perplexity: {ppl:.2f}")
    print(f"Time: {wall/60:.1f} min | Tokens/sec: {toks_per_sec:,.0f}")
    if torch.cuda.is_available():
        print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

if __name__ == "__main__":
    main()


Map:   0%|          | 0/52091 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/52091 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Step,Training Loss,Validation Loss
1000,1.743800,1.607468
2000,1.663600,1.571513
3000,1.621100,1.556442
4000,1.566800,1.555322
5000,1.511700,1.516144
6000,1.507300,1.535158
7000,1.461000,1.531238
8000,1.439400,1.502308
9000,1.411000,1.528601
10000,1.379800,1.524285


[BASELINE] Eval loss: 1.4868 | PPL: 4.42
Wall: 32.8 min | ~Tokens/sec: 83,318
Peak GPU memory: 9.65 GB
